# Advanced Feature Engineering

The first version of the NBA win probability model was able to predict outcomes using basic game-state features such as score differential, time remaining, and momentum.

While the model achieved strong baseline performance, most of its predictive power came from the current score differential. This suggests that additional basketball-specific features are needed to better represent the state of a live game.

This notebook focuses on extracting advanced features from play-by-play data to improve the model's understanding of game context.

The features introduced in this stage aim to represent:

- Which team currently has possession
- Recent scoring trends
- Late-game situations
- Strategic advantages that are not represented by the score alone

In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/nba_pbp/pbp2001.csv")

print(df.shape)
df.head()

(602342, 16)


,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season
0,20000001,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0,0,0,Start of 1st Period (12:13 PM EST),2001
1,20000001,1,PT12M00.00S,0.0,0.0,NYK,948,M. Camby,Jump Ball,NaN,NaN,0,0,0,Jump Ball Camby vs. Ratliff: Tip to Houston,2001
2,20000001,1,PT11M41.00S,0.0,0.0,NYK,84,L. Sprewell,Missed Shot,Jump Shot,Missed,-58,28,6,MISS Sprewell 6' Jump Shot,2001
3,20000001,1,PT11M41.00S,0.0,0.0,PHI,689,T. Ratliff,NaN,NaN,NaN,0,0,0,Ratliff BLOCK (1 BLK),2001
4,20000001,1,PT11M40.00S,0.0,0.0,NaN,1610612755,NaN,Rebound,Unknown,NaN,0,0,0,76ers Rebound,2001


In [28]:
import pandas as pd

full_df = pd.read_parquet("../data/processed/ml_dataset.parquet")
sample_df = pd.read_parquet("../data/processed/ml_dataset_sample.parquet")

print("Full dataset:")
print(full_df.shape)
print(full_df.columns.tolist())

print("\nSample dataset:")
print(sample_df.shape)
print(sample_df.columns.tolist())

Full dataset:
(18255730, 8)
['gameid', 'season', 'period', 'game_seconds_remaining', 'score_diff', 'score_diff_squared', 'momentum', 'home_win']

Sample dataset:
(2000000, 8)
['gameid', 'season', 'period', 'game_seconds_remaining', 'score_diff', 'score_diff_squared', 'momentum', 'home_win']


## Possession Tracking (Paused)

Possession is one of the most important factors in basketball because teams cannot score without controlling the ball.

The original model does not know which team currently has possession, meaning identical game states could receive the same prediction even though one team has an immediate offensive opportunity.

This section develops a possession-tracking feature from play-by-play events and adds it as an input to the model.

In [20]:
sample_game = df[df["gameid"] == df["gameid"].iloc[0]].copy()

sample_game = sample_game.reset_index(drop=True)

sample_game.head()

,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season
0,20000001,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0,0,0,Start of 1st Period (12:13 PM EST),2001
1,20000001,1,PT12M00.00S,0.0,0.0,NYK,948,M. Camby,Jump Ball,NaN,NaN,0,0,0,Jump Ball Camby vs. Ratliff: Tip to Houston,2001
2,20000001,1,PT11M41.00S,0.0,0.0,NYK,84,L. Sprewell,Missed Shot,Jump Shot,Missed,-58,28,6,MISS Sprewell 6' Jump Shot,2001
3,20000001,1,PT11M41.00S,0.0,0.0,PHI,689,T. Ratliff,NaN,NaN,NaN,0,0,0,Ratliff BLOCK (1 BLK),2001
4,20000001,1,PT11M40.00S,0.0,0.0,NaN,1610612755,NaN,Rebound,Unknown,NaN,0,0,0,76ers Rebound,2001


In [21]:
teams = sample_game["team"].dropna().unique()

print(teams)

<ArrowStringArray>
['NYK', 'PHI']
Length: 2, dtype: str


In [22]:
def other_team(current_team, teams):
    if current_team == teams[0]:
        return teams[1]
    else:
        return teams[0]

In [23]:
sample_game["possession"] = None

In [24]:
current_possession = None
last_shot_team = None

In [25]:
current_possession = None

for idx, row in sample_game.iterrows():

    event = row["type"]
    team = row["team"]
    desc = str(row["desc"])


    if event == "Jump Ball":
        current_possession = team


    elif event == "Made Shot":

        last_shot_team = team

        current_possession = other_team(team, teams)


    elif event == "Missed Shot":

        last_shot_team = team


    elif event == "Turnover":
        current_possession = other_team(team, teams)


    elif event == "Rebound":

        rebound_team = None

        if pd.notna(team):
            rebound_team = team

        else:
            desc_lower = desc.lower()

            if "76ers" in desc_lower:
                rebound_team = "PHI"

            elif "knicks" in desc_lower:
                rebound_team = "NYK"

        if rebound_team is not None and last_shot_team is not None:

        
            if rebound_team != last_shot_team:
                current_possession = rebound_team

            else:
                pass


    last_shot_team = None

    sample_game.at[idx, "possession"] = current_possession

In [26]:
sample_game[
    [
        "clock",
        "team",
        "type",
        "desc",
        "possession"
    ]
].head(40)

,clock,team,type,desc,possession
0,PT12M00.00S,NaN,period,Start of 1st Period (12:13 PM EST),None
1,PT12M00.00S,NYK,Jump Ball,Jump Ball Camby vs. Ratliff: Tip to Houston,NYK
2,PT11M41.00S,NYK,Missed Shot,MISS Sprewell 6' Jump Shot,NYK
3,PT11M41.00S,PHI,NaN,Ratliff BLOCK (1 BLK),NYK
4,PT11M40.00S,NaN,Rebound,76ers Rebound,NYK
5,PT11M29.00S,NYK,Foul,Camby S.FOUL (P1.T1),NYK
6,PT11M29.00S,PHI,Free Throw,Ratliff Free Throw 1 of 2 (1 PTS),NYK
7,PT11M29.00S,PHI,Free Throw,MISS Ratliff Free Throw 2 of 2,NYK
8,PT11M28.00S,NYK,Rebound,Ward REBOUND (Off:0 Def:1),NYK
9,PT11M18.00S,PHI,Foul,Ratliff S.FOUL (P1.T1),NYK


In [27]:
sample_game.columns.tolist()

['gameid',
 'period',
 'clock',
 'h_pts',
 'a_pts',
 'team',
 'playerid',
 'player',
 'type',
 'subtype',
 'result',
 'x',
 'y',
 'dist',
 'desc',
 'season',
 'possession']

## Feature Expansion

Possession is the first advanced feature added to the model. Other improvements may include:

- Recent scoring runs
- Team fouls and bonus situations
- Remaining timeouts
- Team strength metrics
- Offensive and defensive efficiency

These features will continue improving the model's ability to represent the complete state of an NBA game.

## Model Performance Comparison

After adding possession, the updated feature set is used to train a new Random Forest model.

Performance is compared against the Version 1 model using:

- Accuracy
- Confusion matrix
- Feature importance
- Brier score
- Probability calibration

The goal is to determine whether additional basketball context improves both prediction accuracy and probability reliability.